# SFJ / Media Skills – GROSSLAUF 2026 (EVT272)

**Einmaliger grosser Produktions-Vorbereitungslauf für CH, DE, AT, LI, USA.** Alle 3’818 vorhandenen Country+ID-Zeilen werden geladen, reconciliert und als einheitlicher Website-Gold-Kandidatenbestand mit einzelnen Evidence-Gates ausgegeben. **Kein Release und keine erfundenen Verifikationen.**

### So funktioniert der Knopfdruck

1. Oben im Menü **Laufzeit → Alle ausführen** wählen; Google-Anmeldung einmal erlauben. Das voreingestellte `GROSSLAUF_KOSTENLOS` führt die Prüfung der **gesamten Population** und den host-schonenden direkten Primärquellenabruf durch. Firecrawl-Kosten **0**.
2. Colab speichert **alle 40 URL-Ergebnisse** als unveränderliche ZIP-Checkpoints in Google Drive (mit SHA-256-Readback). Nach Unterbrüchen erneut **Alle ausführen**: bereits gesicherte Ergebnisse werden wiederverwendet. Kein neuer Hauptdaten-Merge.
3. Optional, erst bei nicht erreichbaren offiziellen Seiten: Modus `MIT_FIRECRAWL` auswählen und **Alle ausführen**; separaten API-Schlüssel im verdeckten Feld eingeben. Maximal **300 neue Scrape-Aufrufe** (nur `markdown`, keine PDF-/LLM-/Agent-Zusätze); vorher zusätzlich API-Key-Kostenlimit im Firecrawl-Dashboard auf höchstens 300 Credits setzen.

### Enthaltene Quellen

- Live-GitHub-PR10-Draft: 1’352 CH/AT/LI-Angebote, Sidecars, B2-140, 111 DE- und 2’355 USA-R4-Zeilen; PR-HEAD und alle Quell-SHA werden beim Start geprüft.
- Autoritative Google-Drive-Projektbibel Band 03 (mindestens EVT272), EVT270 Read-only-Abschlussaudit, EVT272-Feldbrücken, EVT260-Evidence-Judgements und gesicherter offizieller 140er-R2-Snapshot.
- 145 unverändert geschützte historische Reviewfälle und die ursprünglichen QA09-116-IDs werden **nicht** als neue Forschungsarbeit geöffnet. 30 zusammengesetzte USA-Quellzellen bleiben eigene Identitäts-Holds.

### Wichtig

Ein HTTP-200, ein Regex-Fund oder ein alter Owner-Satz bedeutet **nicht** Website-Gold. Der Lauf liefert einen grossen, quellennachvollziehbaren Bestand, eindeutige exakte Feldnachweise soweit vorhanden, eine maschinenlesbare Restfallqueue und einen Website-Importkandidaten mit getrenntem HOLD-Gate. **Keine automatische Veröffentlichung oder stiller Master-Überschreibvorgang.**

**Die im Lauf ausgegebene Google-Drive-Ordneradresse hier im Chat senden.** Damit kann ich den vollständigen Ergebnisbestand direkt übernehmen und die nächsten echten Feld-Korrekturen im Draft samt Test und Readback ausführen.
**Zuverlässiger Start:** Bei Problemen mit dem GitHub-Colab-Link die `.ipynb`-Datei herunterladen, https://colab.research.google.com/ öffnen und **Datei → Notebook hochladen** wählen; danach **Laufzeit → Alle ausführen**.


In [ ]:
#@title 1. Einmaliger grosser Lauf – Startkonfiguration
MODUS = "GROSSLAUF_KOSTENLOS"  #@param ["GROSSLAUF_KOSTENLOS", "AUDIT", "MIT_FIRECRAWL"]
MAX_FREE_URLS = 5000  #@param {type:"integer"}
MAX_FREE_WORKERS = 10  #@param {type:"integer"}
CHECKPOINT_BATCH = 40  #@param {type:"integer"}
FIRECRAWL_MAX_CREDITS = 300  #@param {type:"integer"}
assert MODUS in ("GROSSLAUF_KOSTENLOS", "AUDIT", "MIT_FIRECRAWL")
assert 1<=MAX_FREE_URLS<=5000 and 1<=MAX_FREE_WORKERS<=12
assert 1<=CHECKPOINT_BATCH<=60 and 0<FIRECRAWL_MAX_CREDITS<=300
print("Modus:",MODUS,"· Maximaler kostenloser URL-Batch:",MAX_FREE_URLS)


In [ ]:
%pip -q install requests beautifulsoup4 pandas pymupdf google-api-python-client google-auth-httplib2


In [ ]:
"""SFJ Media Skills: importable, offline-testable helpers for the Colab audit.
No network calls, Drive writes or Firecrawl calls on import.
"""
from __future__ import annotations
import csv
import hashlib
import io
import ipaddress
import json
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import date
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

IN_SCOPE={"schweiz","deutschland","österreich","liechtenstein","usa","vereinigte staaten", "ch","de","at","li","us"}
EMPTY_MARKERS={"", "-", "–", "n/a", "unknown", "nicht bekannt", "nicht publiziert", "nicht veröffentlicht", "keine angabe", "tbd", "k.a.", "nicht_publiziert", "on_request_format", "preisanfrage", "preis auf anfrage"}
MATERIAL_GAPS={"DELIVERY_NOT_VERIFIED","STRUCTURED_PRICE_BUDGET_UNVERIFIED","SOURCE_VERSION_OR_IDENTITY_GUARD","ACCESS_TARIFF_ELIGIBILITY_GUARD","SOURCE_PRIMARY_UNRESOLVED","OWNER_SCOPE_REVIEW","TAX_SCOPE_UNSPECIFIED"}
TRACKING_PARAMS={"fbclid","gclid","mc_cid","mc_eid","igshid"}

def sha(data):
    if isinstance(data,str): data=data.encode('utf-8')
    return hashlib.sha256(data).hexdigest()

def parse_site_js(text):
    start=text.find('[');end=text.rfind(']')
    if start<0 or end<start: raise ValueError('data.js: Angebotsliste nicht erkennbar')
    obj=json.loads(text[start:end+1]);
    if not isinstance(obj,list):raise ValueError('data.js: Liste erwartet')
    ids=[str(x.get('id','')).strip() for x in obj]
    if not all(ids) or len(set(ids))!=len(ids): raise ValueError('Leere oder doppelte Stable-Offer-ID')
    return obj

def parse_sidecar_js(text):
    start=text.find('{');end=text.rfind('}')
    if start<0 or end<start:raise ValueError('Sidecar: JSON fehlt')
    obj=json.loads(text[start:end+1]);
    if not isinstance(obj.get('byId'),dict):raise ValueError('Sidecar byId fehlt')
    return obj

def parse_tsv(text):
    rows=list(csv.DictReader(io.StringIO(text),delimiter='\t'))
    if any(None in r for r in rows):raise ValueError('TSV-Spaltenfehler')
    return rows

def normalize_url(raw):
    """Dedupe only tracking params; preserve business params, cases, route aliases."""
    try:
        s=urlsplit((raw or '').strip())
        if s.scheme.lower()!='https' or not s.hostname:return ''
        h=s.hostname.lower().rstrip('.')
        try:
            ip=ipaddress.ip_address(h)
            if not ip.is_global:return ''
        except ValueError:
            if h=='localhost' or h.endswith('.local') or '.' not in h:return ''
        if s.username or s.password:return ''
        netloc=h+((':'+str(s.port)) if s.port and s.port!=443 else '')
        q=sorted([(k,v) for k,v in parse_qsl(s.query,keep_blank_values=True)
                  if k.lower() not in TRACKING_PARAMS and not k.lower().startswith('utm_')])
        path=s.path or '/'
        return urlunsplit(('https',netloc,path,urlencode(q,doseq=True),''))
    except (ValueError,TypeError):return ''

def missing(v):
    if v is None:return True
    if isinstance(v,str):return v.strip().lower() in EMPTY_MARKERS
    return False

def clean_norm(s):
    s=unicodedata.normalize('NFKD',str(s or ''))
    s=''.join(x for x in s if not unicodedata.combining(x))
    return re.sub(r'[^a-z0-9]+',' ',s.lower()).strip()

def identity_match(title,content):
    """Conservative signal, not proof of exact edition or price."""
    a={x for x in clean_norm(title).split() if len(x)>=4 and x not in {'seminar','kurs','training','journalismus','formation','weiterbildung'}}
    b=set(clean_norm((content or '')[:35000]).split())
    if len(a)<2:return 'UNVERIFIED_SHORT_TITLE'
    fraction=len(a&b)/len(a)
    return 'TITLE_WORDS_PRESENT' if fraction>=0.72 else 'TITLE_MATCH_UNPROVEN'

def date_future(x,today=None):
    today=today or date.today()
    try:return date.fromisoformat(str(x)[:10]) >= today
    except (ValueError,TypeError):return False

def audit_offers(offers,sidecar,b2_rows=None,today=None,closed_ids=None):
    """One row per site ID; never mark whole offer as verified from prose."""
    today=today or date.today()
    byid=sidecar['byId']; expected=set(x['id'] for x in offers)
    if expected!=set(byid):
        raise ValueError(f'Sidecar-Abgleich FEHLER: Website-only={len(expected-set(byid))}, Sidecar-only={len(set(byid)-expected)}')
    b2={r['offer_id']:r for r in (b2_rows or [])}
    qa09_closed=set(closed_ids or [])
    results=[]
    for o in offers:
        id=o['id'];side=byid[id];proof=b2.get(id,{})
        core=[]
        if missing(o.get('title')):core.append('TITLE_MISSING')
        if missing(o.get('provider')):core.append('PROVIDER_MISSING')
        url=normalize_url(o.get('url',''))
        if not url:core.append('HTTPS_SOURCE_MISSING_OR_INVALID')
        if missing(o.get('format')):core.append('FORMAT_NOT_PUBLISHED')
        if missing(o.get('duration')):core.append('DURATION_NOT_PUBLISHED')
        if missing(o.get('priceDisplay')):core.append('PRICE_TEXT_NOT_PUBLISHED')
        if missing(o.get('language')):core.append('LANGUAGE_NOT_PUBLISHED')
        if missing(o.get('access')):core.append('ACCESS_TEXT_NOT_PUBLISHED')
        if missing(o.get('description')):core.append('DESCRIPTION_NOT_PUBLISHED')
        po=side.get('priceOptions') or []
        if not side.get('priceEligible',False) or not po or all(p.get('amountType') in ('NOT_PUBLISHED','ON_REQUEST',None) for p in po):
            core.append('STRUCTURED_PRICE_NOT_VERIFIED')
        delivery=side.get('delivery') or {}
        if delivery.get('evidenceTier')!='VERIFIED':core.append('DELIVERY_NOT_VERIFIED')
        tags=[v for v in str(proof.get('field_gap_tags','')).split('|') if v]
        critical=sorted(set(core+list(set(tags)&MATERIAL_GAPS)))
        sessions=o.get('sessions',None)
        future_count=sum(1 for sess in sessions if date_future(sess.get('startDate'),today)) if isinstance(sessions,list) else 0
        is_history=(isinstance(sessions,list) and len(sessions)>0 and future_count==0)
        # An ended individual course is archival, not a mandate to relaunch discovery.
        # A verified undated self-study channel stays relevant even if its old classroom sessions ended.
        verified_selfstudy=delivery.get('evidenceTier')=='VERIFIED' and any(
            v.get('temporalMode')=='ASYNCHRONOUS' and 'SELF_STUDY' in (v.get('instructionFormats') or [])
            for v in (delivery.get('variants') or []))
        if is_history and verified_selfstudy:is_history=False
        if not isinstance(sessions,list):
            end=o.get('endDate') or o.get('startDate')
            if end and not date_future(end,today):is_history=True
        allowed_country=clean_norm(o.get('country')) in {clean_norm(x) for x in IN_SCOPE}
        interesting=bool(critical) and bool(url) and not is_history and allowed_country and id not in qa09_closed
        weight=(80 if 'SOURCE_VERSION_OR_IDENTITY_GUARD' in critical else 0) + (75 if 'OWNER_SCOPE_REVIEW' in critical else 0)
        weight+=(40 if 'DELIVERY_NOT_VERIFIED' in critical else 0)+(35 if 'STRUCTURED_PRICE_NOT_VERIFIED' in critical or 'STRUCTURED_PRICE_BUDGET_UNVERIFIED' in critical else 0)
        weight+=(20 if 'ACCESS_TARIFF_ELIGIBILITY_GUARD' in critical else 0)
        weight+=min(24,len(critical)*4)
        if proof.get('priority','').startswith('P0'):weight+=25
        results.append(dict(offer_id=id,provider=o.get('provider',''),title=o.get('title',''),country=o.get('country',''),source_url=url,
                            existing_start=o.get('start',''),session_count=len(sessions) if isinstance(sessions,list) else '',future_session_count=future_count,
                            currentness='ARCHIVE_NO_FUTURE_SESSION' if is_history else 'ACTIVE_OR_UNDATED',
                            price_eligible=bool(side.get('priceEligible')),delivery_tier=delivery.get('evidenceTier','UNKNOWN'),
                            source_hold_tags='|'.join(tags),open_material_fields='|'.join(critical),
                            priority_score=weight,requires_source_research=interesting,
                            previous_publication_status=o.get('publicationStatus',''),existing_source_key=sha(url) if url else '',
                            decision='QA09_CLOSED_REUSE_ONLY' if id in qa09_closed else ('SOURCE_RESEARCH_CANDIDATE' if interesting else ('ARCHIVE_NO_AUTO_RESCAN' if is_history else 'NO_PAID_FETCH_NEEDED'))))
    return results

def source_tasks(audit):
    grouped=defaultdict(lambda:{'ids':[],'providers':set(),'score':0,'fields':set()})
    for a in audit:
        if not a['requires_source_research'] or not a['source_url']:continue
        g=grouped[a['source_url']]
        g['ids'].append(a['offer_id']);g['providers'].add(a['provider']);g['score']=max(g['score'],a['priority_score'])
        g['fields'].update(a['open_material_fields'].split('|'))
    tasks=[]
    for url,g in grouped.items():
        tasks.append(dict(url=url,ids=sorted(g['ids']),providers=sorted(g['providers']),priority=g['score']+min(len(g['ids']),12)*3,
                          missing_fields=sorted(g['fields']-set([''])),domain=urlsplit(url).hostname))
    # interleave domains to avoid 300 URLs from one provider / host, preserve priority within host
    hosts=defaultdict(list)
    for t in tasks:hosts[t['domain']].append(t)
    for z in hosts.values():z.sort(key=lambda x:(-x['priority'],x['url']))
    out=[]
    while any(hosts.values()):
        for host in sorted(hosts,key=lambda h:(-(hosts[h][0]['priority'] if hosts[h] else -1),h)):
            if hosts[host]:out.append(hosts[host].pop(0))
    return out

KEYWORD_RE={
  'PRICE':re.compile(r'(?i)(?<!\w)(?:CHF|EUR|USD|€|\$)\s*[0-9][0-9\s\x27.,]*'),
  'DATES':re.compile(r'(?i)\b\d{1,2}[./]\d{1,2}[./]\d{2,4}\b|\b20\d{2}-\d{2}-\d{2}\b'),
  'DURATION':re.compile(r'(?i)\b\d{1,3}\s*(?:tage|tagen|days?|jours?|stunden|hours?|heures?|wochen|weeks?|semesters?)\b'),
  'DELIVERY':re.compile(r'(?i)\b(?:online.?live|online|hybrid|pr[aä]senz|pr[eé]sentiel|distanciel|in.person|webinar|self.paced|selbststudium)\b'),
  'ACCESS':re.compile(r'(?i)\b(?:zulassung|voraussetzungen|admission|eligibility|inscription|anmeldung|bewerbung)\b')
}

def find_evidence(title,full_text,url):
    """Find source snippets only; no source-derived value becomes trusted by regex."""
    status=identity_match(title,full_text)
    lines=[re.sub(r'\s+',' ',s).strip() for s in (full_text or '').splitlines()]
    snippets=[]
    for field,rx in KEYWORD_RE.items():
        matching=[]
        for line in lines:
            if len(line)<5 or len(line)>800:continue
            if rx.search(line):matching.append(line[:420])
        for line in matching[:5]:
            snippets.append(dict(field=field,source_url=url,excerpt=line,identity_signal=status,
                                 decision='EVIDENCE_ONLY_REQUIRES_FIELD_AND_EDITION_ADJUDICATION'))
    return snippets


def compact_stats(audit,tasks):
    c=Counter(a['decision'] for a in audit)
    p=Counter(x for a in audit for x in a['open_material_fields'].split('|') if x)
    return {'offers':len(audit),'unique_urls_research_candidates':len(tasks),'by_decision':dict(c),'most_common_gaps':p.most_common(18),
            'delivery_not_verified':sum(a['delivery_tier']!='VERIFIED' for a in audit),
            'structured_price_eligible':sum(a['price_eligible'] for a in audit),
            'historical_offers':sum(a['currentness']=='ARCHIVE_NO_FUTURE_SESSION' for a in audit)}


def csv_bytes(rows,fields=None):
    fields=fields or (list(rows[0]) if rows else ['empty'])
    io_obj=io.StringIO(newline='');w=csv.DictWriter(io_obj,fieldnames=fields,delimiter='\t',extrasaction='ignore')
    w.writeheader();
    for row in rows:w.writerow({k:json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list)) else v for k,v in row.items()})
    return io_obj.getvalue().encode('utf-8')


def decode_jsonlines(blob):
    return [json.loads(line) for line in blob.decode("utf-8").splitlines() if line.strip()]

def verified_zip_bytes(payload, required):
    import zipfile, io
    with zipfile.ZipFile(io.BytesIO(payload)) as f:
        if f.testzip() is not None:raise RuntimeError("ZIP CRC Fehler")
        if not set(required).issubset(set(f.namelist())):raise RuntimeError("Ein notwendiges Evidence-Dokument fehlt")
        return {n:f.read(n) for n in f.namelist()}

def multi_source(raw):
    """No guessed one-URL assignment for combined owner source cells."""
    if not isinstance(raw,str):return []
    if ';' in raw or ' | ' in raw or ' + ' in raw:
        return [normalize_url(v) for v in re.split(r'\s*(?:;|\|)\s*|\s+\+\s+',raw) if normalize_url(v)]
    one=normalize_url(raw)
    return [one] if one else []

def source_group_tasks(website_rows,stage_rows,offers,qa09_closed,locked_ids,as_of):
    """Deterministic one-pass full-population planning with protected IDs excluded."""
    legacy={o['id']:o for o in offers}
    group={}
    stats=Counter()
    def add(url,market,id,title,kind,priority,source_status):
        norm=normalize_url(url)
        if not norm:return
        entry=group.setdefault(norm,{'url':norm,'ids':set(),'markets':set(),'priority':0,'source_classes':set(),'titles':set()})
        entry['ids'].add(f'{market}:{id}');entry['markets'].add(market)
        entry['titles'].add(title[:120]);entry['priority']=max(priority,entry['priority']);entry['source_classes'].add(source_status)
    for r in website_rows:
        o=legacy[r['id']];id=r['id'];market=r['market'];policy=r['reviewPolicy']
        if id in qa09_closed or id in locked_ids or r['protectedNoReopen']:
            stats['protected_catalogue']+=1;continue
        if r.get('kind')!='LEGACY_STABLE_OFFER':continue
        if policy=='EVT270_HISTORICAL_NO_REOPEN_OUTSIDE_B2':continue
        # An archived one-time event without a valid future occurrence is not re-discovered.
        sessions=o.get('sessions')
        if isinstance(sessions,list) and sessions and not any(str(s.get('startDate',''))>=as_of for s in sessions):
            stats['archived_no_future_session']+=1;continue
        if not sessions and o.get('endDate') and o.get('endDate')<as_of:
            stats['archived_ended']+=1;continue
        if policy=='B2_EXISTING_EVIDENCE_FIRST':priority=100
        elif policy=='SIX_EXCEPTION_EXISTING_EVIDENCE_FIRST':priority=95
        else:priority=40
        add(o['url'],market,id,o['title'],'offer',priority,policy)
    for r in stage_rows:
        if r['kind']=='OCCURRENCE_ONLY_DO_NOT_PROMOTE':
            stats['stage_occurrence_not_promoted']+=1;continue
        if r['sourceUrlStatus']!='SINGLE_SOURCE_URL_UNVERIFIED':
            stats['multi_url_identity_hold']+=1;continue
        if 'ARCHIVED' in str(r.get('objectType','')).upper():
            stats['stage_archived_not_refetched']+=1;continue
        add(r['url'],r['market'],r['id'],r['title'],'staged',25,r['sourceUrlStatus'])
    tasks=[]
    for g in group.values():
        tasks.append({**g,'ids':sorted(g['ids']),'markets':sorted(g['markets']),
          'source_classes':sorted(g['source_classes']),'titles':sorted(g['titles']),
          'domain':urlsplit(g['url']).hostname})
    # Interleave by host for politeness; priority within each host.
    hosts=defaultdict(list)
    for t in tasks:hosts[t['domain']].append(t)
    for x in hosts.values():x.sort(key=lambda y:(-y['priority'],y['url']))
    out=[]
    while any(hosts.values()):
        for host in sorted(hosts, key=lambda h:(-(hosts[h][0]['priority'] if hosts[h] else -1),h)):
            if hosts[host]:out.append(hosts[host].pop(0))
    return out,dict(stats)


In [ ]:
#@title 3. Control Plane laden, 3’818er Bestand zusammenführen, SHA-Gate
import io, os, json, zipfile, hashlib, time, re, threading, getpass
from collections import Counter
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlsplit
from urllib.robotparser import RobotFileParser
import requests
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload
from IPython.display import display, Markdown

REPO='george-sarpong/weiterbildungsradar-journalismus'
PR_NO=10
BIBLE_ID='1b1Zx5sjZuqYrEQr6zy9xlxs0eW7R4iDOpEBFl8ZeXsw'
QA09_ROSTER_ID='1kvL0WkdffjOB0BytsP66nuwRpEbjgmfZQPAblowkZGY'
EVT271_ZIP_ID='1IhK80RxL9lFolHgJZhe1OJTGqbdkb1Jm'
EVT270_ZIP_ID='1I6gKhrqB3hYRgETQorlyOT5GAypfXOsT'
EVT260_ZIP_ID='1tPaYbzfX18_jGF8d_67ZQ50C8Y0Mlnno'
B2_R2_ZIP_ID='1_3DNVDaiBaPdm9co2fpiDFuEKm-rQbK6'
USER_AGENT='SFJ-MediaSkills-Research/1.1 (official data audit; no automated publication)'
PAID_URL='https://api.firecrawl.dev/v2/scrape'
CREDIT_URL='https://api.firecrawl.dev/v2/team/credit-usage'
auth.authenticate_user()
credentials,_=google.auth.default()
drive_api=build('drive','v3',credentials=credentials,cache_discovery=False)
docs_api=build('docs','v1',credentials=credentials,cache_discovery=False)

def remote_bytes(file_id):
    buffer=io.BytesIO();media=drive_api.files().get_media(fileId=file_id)
    downloader=MediaIoBaseDownload(buffer,media,chunksize=1024*1024)
    done=False
    while not done:_,done=downloader.next_chunk(num_retries=2)
    return buffer.getvalue()

def iter_doc_text(obj):
    if isinstance(obj,dict):
        if 'textRun' in obj and isinstance(obj['textRun'],dict):yield obj['textRun'].get('content','')
        for k,v in obj.items():
            if k!='textRun':yield from iter_doc_text(v)
    elif isinstance(obj,list):
        for v in obj:yield from iter_doc_text(v)

project_doc=docs_api.documents().get(documentId=BIBLE_ID,includeTabsContent=True).execute()
bible_text=''.join(iter_doc_text(project_doc))
# EVT272 introduced this notebook; never reject the current authorised successor.
# Later successors are safe for this read-only pipeline only if source SHA checks
# against the EVT271 baseline (below) still pass. No master/website writes here.
newer=[int(n) for n in re.findall(r'EVT-BIBEL-20261107-(\d+)',bible_text)]
if not newer or max(newer)<272:
    raise RuntimeError('Projektbibel mindestens EVT272 nicht gefunden – kein Quellenlauf')
print('Drive-Control-Plane-Successor:',max(newer),'– Feldquellen-SHA-Abgleich folgt')
if max(newer)>272:
    print('Neuerer Drive-Successor vorhanden; dieser reine Recherchelauf prüft alle Baseline-SHA-Hashes und verändert keine autoritativen Daten.')
qa_doc=docs_api.documents().get(documentId=QA09_ROSTER_ID,includeTabsContent=True).execute()
qa09_text=''.join(iter_doc_text(qa_doc))
qa09_closed={m.group(2).strip() for m in re.finditer(r'(?:^|\n)\s*(\d{3})\s*\|\s*([^|\n]+?)\s*\|',qa09_text)}
if len(qa09_closed)!=116:raise RuntimeError('QA09 Freeze 116 ID GATE nicht erfüllt')

response=requests.get(f'https://api.github.com/repos/{REPO}/pulls/{PR_NO}',timeout=35,headers={'Accept':'application/vnd.github+json'})
response.raise_for_status();pr=response.json()
if pr.get('state')!='open' or not pr.get('draft') or pr.get('merged'):
    raise RuntimeError('PR10 nicht mehr offen/unveröffentlicht: Lauf gesperrt')
HEAD=pr['head']['sha']

def github_raw(filename):
    resp=requests.get(f'https://raw.githubusercontent.com/{REPO}/{HEAD}/{filename}',timeout=90,headers={'User-Agent':USER_AGENT})
    resp.raise_for_status()
    return resp.content.decode('utf-8')

base_files={
 'offers':github_raw('data.js'),
 'sidecars':github_raw('smart-search-data.js'),
 'deUs':github_raw('data/staging/target5-de-usa-wave1-r4-20260915.json'),
 'usEvidence':github_raw('data/staging/usa-owner-evidence-crosswalk-20261107.jsonl'),
 'b2':github_raw('docs/b2/evt264-active-140-website-gold-field-gates-20261107.tsv'),
 'policy':github_raw('data/staging/evt270-currentness-policy-20261107.json')
}
zip271=verified_zip_bytes(remote_bytes(EVT271_ZIP_ID),[
 'manifest.json','catalogue-field-bridge-1352.jsonl','de-usa-owner-product-bridge-2466.jsonl',
 'unprotected-currentness-117.jsonl','locked-currentness-145.jsonl','city-only-owner-21.jsonl'])
manifest271=json.loads(zip271['manifest.json'])
for k,content in base_files.items():
    if sha(content)!=manifest271['sources'][k]:raise RuntimeError('Source SHA drift: '+k)
for k,expected in manifest271['outputs'].items():
    if sha(zip271[k])!=expected:raise RuntimeError('Output SHA drift: '+k)
bridge_catalogue=decode_jsonlines(zip271['catalogue-field-bridge-1352.jsonl'])
bridge_stage=decode_jsonlines(zip271['de-usa-owner-product-bridge-2466.jsonl'])
bridge_due=decode_jsonlines(zip271['unprotected-currentness-117.jsonl'])
bridge_locked=decode_jsonlines(zip271['locked-currentness-145.jsonl'])
bridge_cities=decode_jsonlines(zip271['city-only-owner-21.jsonl'])
zip270=verified_zip_bytes(remote_bytes(EVT270_ZIP_ID),['547_FIELD_EVIDENCE_DECISIONS.jsonl','262_CURRENTNESS_POLICY_FILTERED.jsonl'])
zip260=verified_zip_bytes(remote_bytes(EVT260_ZIP_ID),['ALL_547_SUCCESSOR_SOURCE_DECISIONS.csv'])
zipR2=verified_zip_bytes(remote_bytes(B2_R2_ZIP_ID),['source-signals.jsonl','source-summary.json'])
legacy_offers=parse_site_js(base_files['offers'])
sidecar=parse_sidecar_js(base_files['sidecars'])
stage=json.loads(base_files['deUs'])['records']
policy=json.loads(base_files['policy'])
if not (len(legacy_offers)==1352 and len(bridge_catalogue)==1352 and len(bridge_stage)==2466
        and len(bridge_due)==117 and len(bridge_locked)==145 and len(bridge_cities)==21
        and len(stage)==2466 and len(qa09_closed)==116):
    raise RuntimeError('3818/117/145/21 Baseline geändert – keine alte Quelle überschreiben')
if len({r['id'] for r in bridge_catalogue})!=1352 or len({r['market']+':'+r['id'] for r in bridge_stage})!=2466:
    raise RuntimeError('COUNTRY+ID UNIQUE Gate fehlgeschlagen')
locked_ids={r['id'] for r in bridge_locked}
if len(locked_ids)!=145:raise RuntimeError('NO_REOPEN_LOCKED_ID_DRIFT')
if {r['id'] for r in bridge_due}&locked_ids:raise RuntimeError('114/145 Protection overlap')
if len(policy.get('protectedB2',[]))!=29:raise RuntimeError('Historischer 29er B2 Schutz nicht gefunden')

# Absolute date stamps only for runtime checks; original historical records remain unchanged.
TODAY=datetime.now(ZoneInfo('Europe/Zurich')).date()
AFO={o['id']:o for o in legacy_offers}
AFS=sidecar['byId']
source_work,source_stats=source_group_tasks(bridge_catalogue,bridge_stage,legacy_offers,qa09_closed,locked_ids,TODAY.isoformat())
if len(source_work)>MAX_FREE_URLS:
    print('WARNUNG: mehr eindeutige offizielle URLs als Abrufkappe. Audit bleibt vollständig, Abrufe werden geteilt.')

all_3818=[]
city_by_id={r['id']:r for r in bridge_cities}
for row in bridge_catalogue:
    original=AFO[row['id']];side=AFS[row['id']]
    city=city_by_id.get(row['id'])
    all_3818.append({**row,'catalogueOriginal':original,'priceModel':side['priceOptions'],
     'cityOwnerEvidence':city,'websiteGold':'NOT_PROVEN' if row['websiteGold']=='NOT_PROVEN' else 'NOT_ASSESSED',
     'releaseGate':'HOLD_NO_AUTOMATIC_RELEASE',
     'sourceEvidenceStatus':'OLD_FIELDS_AND_EXACT_CITY_IF_DOCUMENTED_NOT_FRESH_SOURCE'})
for row in bridge_stage:
    all_3818.append({**row,'releaseGate':'HOLD_NO_AUTOMATIC_RELEASE',
     'sourceEvidenceStatus':'HISTORICAL_OWNER_ONLY_NOT_WEBSITE_GOLD'})
assert len(all_3818)==3818

core_report={
 'phase':'PRODUCTION_BASE_CONSOLIDATION_NOT_RELEASE',
 'country_ids':3818,'existing_offers':1352,'stage_de':111,'stage_usa':2355,
 'qa09_no_reopen':116,'locked_due_no_reopen':145,'eligible_due':117,
 'city_owner_evidence':len(bridge_cities),'historical_evidence_groups':2123,'reviewed_field_groups':547,
 'composite_usa_urls':sum(r['sourceUrlStatus']!='SINGLE_SOURCE_URL_UNVERIFIED' for r in bridge_stage),
 'unique_source_urls_planned':len(source_work),'source_exclusions':source_stats,
 'source_commit':HEAD,'source_hashes':manifest271['sources'],'drive_bible_revision':project_doc.get('revisionId'),
 'data_profile':'READ_ONLY_EVIDENCE_GATED','published':False,
 'as_of':TODAY.isoformat()}
print('FÜNF-LÄNDER-BASIS VOLLSTÄNDIG RECONCILIERT:')
print(json.dumps(core_report,ensure_ascii=False,indent=2))
display(pd.DataFrame([{'Gesamtdatensätze':3818,'CH/AT/LI':1352,'DE':111,'USA':2355,
                       'Recherche-URLs':len(source_work),'Schutzfälle':145}]))


def folder_by_name(name):
    q="name = '%s' and mimeType = 'application/vnd.google-apps.folder' and trashed = false" % name.replace("'","\\'")
    fs=drive_api.files().list(q=q,fields='files(id,name)',page_size=100).execute().get('files',[])
    if len(fs)>1:raise RuntimeError('Mehrere gleichnamige Arbeitsordner')
    if fs:return fs[0]['id']
    return drive_api.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder'},fields='id').execute()['id']

# Stable by SOURCE hashes, not by code commit, so check-point reuse survives code-only changes.
RUN_KEY=sha(json.dumps(manifest271['sources'],sort_keys=True))[:16]
FOLDER=f'SFJ_GROSSLAUF_TARGET5_{RUN_KEY}'
folder_id=folder_by_name(FOLDER)
print('Laufordner:',f'https://drive.google.com/drive/folders/{folder_id}')

def find_file(name):
    files=drive_api.files().list(q="name = '%s' and '%s' in parents and trashed = false"%(name.replace("'","\\'"),folder_id),
       fields='files(id,name)',page_size=100).execute().get('files',[])
    if len(files)>1:raise RuntimeError('Doppelter Dateiname im Checkpoint-Ordner: '+name)
    return files[0] if files else None

def write_immutable(name,payload):
    if isinstance(payload,str):payload=payload.encode('utf-8')
    expected=sha(payload);f=find_file(name)
    if f:
        if sha(remote_bytes(f['id']))!=expected:raise RuntimeError('Drive-Hash Konflikt: '+name)
        return f['id']
    sent=MediaIoBaseUpload(io.BytesIO(payload),mimetype='application/zip',resumable=False)
    id=drive_api.files().create(body={'name':name,'parents':[folder_id]},media_body=sent,fields='id').execute()['id']
    if sha(remote_bytes(id))!=expected:raise RuntimeError('Drive-Readback falsch: '+name)
    return id

def make_zip(files):
    buffer=io.BytesIO()
    with zipfile.ZipFile(buffer,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
        for name,data in files.items():
            entry=zipfile.ZipInfo(name,date_time=(2026,11,7,0,0,0))
            entry.compress_type=zipfile.ZIP_DEFLATED
            z.writestr(entry,data if isinstance(data,bytes) else data.encode('utf-8'))
    return buffer.getvalue()

def jsonl_bytes(rows):return ('\n'.join(json.dumps(x,ensure_ascii=False) for x in rows)+'\n').encode('utf-8')

# All 3818 rows are persisted first; source fetch is next and never affects the raw master.
base_zip=make_zip({
 'alle_3818_daten_kandidaten.jsonl':jsonl_bytes(all_3818),
 'source_workplan.jsonl':jsonl_bytes(source_work),
 'currentness_zulaessig_117.jsonl':jsonl_bytes(bridge_due),
 'historisch_gesperrt_145.jsonl':jsonl_bytes(bridge_locked),
 'stadt_nachweise_21.jsonl':jsonl_bytes(bridge_cities),
 'evt260_547_belegentscheide.csv':zip260['ALL_547_SUCCESSOR_SOURCE_DECISIONS.csv'],
 'evt270_547_belegentscheide.jsonl':zip270['547_FIELD_EVIDENCE_DECISIONS.jsonl'],
 'evt268_r2_quellensignale.jsonl':zipR2['source-signals.jsonl'],
 'manifest.json':json.dumps(core_report,ensure_ascii=False,indent=2)})
base_file='KONSOLIDIERTER_GRUNDSTOCK_'+RUN_KEY+'_'+sha(base_zip)[:12]+'.zip'
write_immutable(base_file,base_zip)
print('Gesamter 3818er Grundstock dauerhaft in Drive gespeichert:',base_file)


In [ ]:
#@title 4. Primärquellen grosser Batch – kostenlos; Firecrawl nur optional
# Der Gesamtdatensatz ist bereits vollständig in Drive persistiert. Dieser Schritt aktualisiert nur Quellen-Evidenz.
if MODUS == 'AUDIT':
    print('AUDIT fertig: kostenlos, keine zusätzlichen Anbieter-HTTP-Aufrufe.')
else:
    print('Direkter Quelllauf: deduplizierte, host-schonende Parallelverarbeitung.')
    def list_work_files():
        found=[];token=None
        while True:
            x=drive_api.files().list(q=f"'{folder_id}' in parents and trashed = false",
                fields='nextPageToken,files(id,name)',page_size=1000,page_token=token).execute()
            found.extend(x.get('files',[]));token=x.get('nextPageToken')
            if not token:break
        return found
    def zip_jsonl(blob):
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            if 'results.jsonl' not in z.namelist():return []
            return decode_jsonlines(z.read('results.jsonl'))
    files=list_work_files();free_results={};paid_results={};reservations={}
    for f in files:
        n=f['name'];prefix=n.split('_')[0]
        if prefix in ('FREE','PAID','RESERVE') and ('_'+RUN_KEY+'_') in n and n.endswith('.zip'):
            records=zip_jsonl(remote_bytes(f['id']))
            for rec in records:
                if prefix=='FREE':free_results[rec['url']]=rec
                elif prefix=='PAID':paid_results[rec['url']]=rec
                else:reservations[rec['url']]=rec
    print('Wiederaufnahme:',len(free_results),'Gratisquellen;',len(paid_results),'Firecrawl-Ergebnisse;',len(reservations),'reserviert.')

    ROBOT_LOCK=threading.Lock();ROBOTS={};ROBOT_HOST_LOCKS={}
    def robot_allowed(url):
        origin=f'{urlsplit(url).scheme}://{urlsplit(url).netloc}'
        with ROBOT_LOCK:
            lock=ROBOT_HOST_LOCKS.setdefault(origin,threading.Lock())
        # Per-host lock: provider robots fetches run in parallel, without duplicate same-host calls.
        with lock:
            if origin in ROBOTS:return ROBOTS[origin].can_fetch(USER_AGENT,url) if ROBOTS[origin] else True
            try:
                rr=requests.get(origin+'/robots.txt',timeout=(6,12),headers={'User-Agent':USER_AGENT})
                if rr.status_code==200:
                    rp=RobotFileParser();rp.parse(rr.text.splitlines());ROBOTS[origin]=rp
                    return rp.can_fetch(USER_AGENT,url)
                if rr.status_code in (401,403):
                    rp=RobotFileParser();rp.disallow_all=True;ROBOTS[origin]=rp;return False
                ROBOTS[origin]=None;return True
            except requests.RequestException:
                ROBOTS[origin]=None;return False
    HOST_LOCK=threading.Lock();HOST_NEXT={}
    def host_pace(url):
        host=urlsplit(url).hostname
        with HOST_LOCK:
            delay=max(0,HOST_NEXT.get(host,0)-time.monotonic())
            HOST_NEXT[host]=max(time.monotonic(),HOST_NEXT.get(host,0))+1.0
        if delay:time.sleep(delay)
    def free_fetch(t):
        url=t['url'];out={'url':url,'method':'FREE_DIRECT','status':'UNKNOWN',
            'fetched_at':datetime.now(timezone.utc).isoformat(),'body':'','body_sha256':'',
            'offer_ids':t['ids'],'metadata_status':'','note':''}
        if not robot_allowed(url):
            out.update(status='ROBOTS_NOT_ALLOWED_OR_UNKNOWN',note='robots.txt unklar oder Sperre; kein Bypass');return out
        host_pace(url)
        try:
            with requests.get(url,timeout=(9,24),stream=True,allow_redirects=True,headers={
                'User-Agent':USER_AGENT,'Accept':'text/html,application/pdf,text/plain,*/*'}) as resp:
                out['metadata_status']=resp.status_code
                if not normalize_url(resp.url):out['status']='REDIRECT_INVALID';return out
                if resp.status_code!=200:out['status']='HTTP_'+str(resp.status_code);return out
                ctype=resp.headers.get('Content-Type','').lower();chunks=[];total=0
                for chunk in resp.iter_content(chunk_size=65536):
                    total+=len(chunk)
                    if total>6_000_000:out['status']='CONTENT_TOO_LARGE';return out
                    chunks.append(chunk)
                raw=b''.join(chunks)
                if ctype.startswith('application/pdf') or raw[:4]==b'%PDF':
                    import fitz
                    pdf=fitz.open(stream=raw,filetype='pdf')
                    text='\n'.join(p.get_text() for p in pdf[:min(len(pdf),60)])
                    out['method']='FREE_PDF'
                elif 'html' in ctype or b'<html' in raw[:3000].lower():
                    soup=BeautifulSoup(raw,'html.parser')
                    for bad in soup(['script','style','noscript','nav','footer']):bad.decompose()
                    main=soup.select_one('main, article, [role=main]') or soup.body or soup
                    text=main.get_text('\n',strip=True)
                else:text=raw.decode('utf-8',errors='replace')
                text=text[:85000]
                out.update(body=text,body_sha256=sha(text),status='CONTENT_OK' if len(text)>=220 else 'CONTENT_INSUFFICIENT')
                if not text.strip():out['status']='EMPTY_TEXT'
        except Exception as e:
            out.update(status='FETCH_FAILED',note=type(e).__name__+': '+str(e)[:140])
        return out

    remaining=[t for t in source_work if t['url'] not in free_results]
    if len(remaining)>MAX_FREE_URLS:
        print('Abruf-Kappe erreicht:',MAX_FREE_URLS,'von',len(remaining),'ausstehenden Quellen; Rest bleibt als Queue erhalten.')
        remaining=remaining[:MAX_FREE_URLS]
    print('Deduplizierte Primärquellen:',len(source_work),' · noch ausstehend:',len(remaining))
    for start in range(0,len(remaining),CHECKPOINT_BATCH):
        batch=remaining[start:start+CHECKPOINT_BATCH]
        with ThreadPoolExecutor(max_workers=MAX_FREE_WORKERS) as pool:
            results=list(pool.map(free_fetch,batch))
        batch_key=sha('\n'.join(sorted(t['url'] for t in batch)))[:14]
        name=f'FREE_{RUN_KEY}_{batch_key}.zip'
        write_immutable(name,make_zip({'results.jsonl':jsonl_bytes(results)}))
        free_results.update({r['url']:r for r in results})
        print('Gesichert:',len(free_results),'/',len(source_work),'kostenlose URLs')

    credits_at_start=credits_at_end=None
    if MODUS=='MIT_FIRECRAWL':
        print('Firecrawl nur für nach direkten Abrufen nicht ausreichende Quellen; bitte separaten Key mit eigenem 300er-Ausgabenlimit benutzen.')
        key=getpass.getpass('Firecrawl API-Key (verdeckt, RAM only): ').strip()
        if not key.startswith('fc-'):raise RuntimeError('Ungültiger Firecrawl-Key; keine bezahlte Abfrage')
        auth_headers={'Authorization':'Bearer '+key,'Content-Type':'application/json'}
        def credits():
            resp=requests.get(CREDIT_URL,headers=auth_headers,timeout=25);resp.raise_for_status()
            v=resp.json()
            if not v.get('success') or 'remainingCredits' not in v.get('data',{}):
                raise RuntimeError('Firecrawl-Saldo nicht validierbar – Kostenlauf gesperrt')
            return float(v['data']['remainingCredits'])
        credits_at_start=credits()
        budget=max(0,min(FIRECRAWL_MAX_CREDITS-len(reservations),int(credits_at_start-30)))
        paid_targets=[t for t in source_work if t['url'] in free_results
            and free_results[t['url']]['status'] in ('CONTENT_INSUFFICIENT','EMPTY_TEXT','HTTP_403','FETCH_FAILED')
            and t['url'] not in paid_results and t['url'] not in reservations]
        paid_targets=paid_targets[:budget]
        print('Neue Firecrawl-Aufrufe maximal:',len(paid_targets),'von',budget,'Budget; alles andere bleibt Restfall.')
        PAID_LOCK=threading.Lock();PAID_NEXT=[time.monotonic()]
        def paid_pace():
            with PAID_LOCK:
                wait=max(0,PAID_NEXT[0]-time.monotonic());PAID_NEXT[0]=max(PAID_NEXT[0],time.monotonic())+6.1
            if wait:time.sleep(wait)
        def paid_fetch(t):
            paid_pace();out={'url':t['url'],'method':'FIRECRAWL_MARKDOWN','status':'UNKNOWN',
                'fetched_at':datetime.now(timezone.utc).isoformat(),'body':'','body_sha256':'',
                'offer_ids':t['ids'],'metadata_status':'','note':''}
            try:
                resp=requests.post(PAID_URL,headers=auth_headers,json={'url':t['url'],
                    'formats':['markdown'],'onlyMainContent':True,'timeout':30000},timeout=65)
                out['metadata_status']=resp.status_code
                if resp.status_code!=200:out.update(status='API_'+str(resp.status_code),note=resp.text[:120]);return out
                raw=resp.json().get('data') or resp.json()
                text=(raw.get('markdown') or '')[:85000]
                sc=(raw.get('metadata') or {}).get('statusCode')
                out.update(body=text,body_sha256=sha(text),status='CONTENT_OK' if len(text)>=220 and (sc in (None,200)) else 'CONTENT_INSUFFICIENT')
            except Exception as e:out.update(status='PAID_REQUEST_UNCERTAIN',note=type(e).__name__)
            return out
        for start in range(0,len(paid_targets),2):
            batch=paid_targets[start:start+2]
            token=sha('\n'.join(sorted(t['url'] for t in batch)))[:14]
            reserve=f'RESERVE_{RUN_KEY}_{token}.zip';paid=f'PAID_{RUN_KEY}_{token}.zip'
            write_immutable(reserve,make_zip({'results.jsonl':jsonl_bytes([{'url':t['url'],'reserved_at':datetime.now(timezone.utc).isoformat()} for t in batch])}))
            reservations.update({t['url']:t for t in batch})
            with ThreadPoolExecutor(max_workers=2) as pool:got=list(pool.map(paid_fetch,batch))
            write_immutable(paid,make_zip({'results.jsonl':jsonl_bytes(got)}))
            paid_results.update({r['url']:r for r in got})
            print('Firecrawl reserviert/gesichert:',len(reservations),'/',FIRECRAWL_MAX_CREDITS)
        credits_at_end=credits();key=None;auth_headers=None
        print('Team-Saldo vor/nach:',credits_at_start,credits_at_end,'(kein Beleg für fremde gleichzeitige Nutzung).')

    byurl={t['url']:t for t in source_work}
    best=dict(free_results)
    for u,r in paid_results.items():
        if r['status']=='CONTENT_OK' or u not in best or best[u]['status']!='CONTENT_OK':best[u]=r
    candidates=[];left=[];source_status=[]
    id_lookup={f'{x["market"]}:{x["id"]}':x for x in all_3818}
    for t in source_work:
        result=best.get(t['url'])
        if not result:
            left.append({'url':t['url'],'ids':t['ids'],'reason':'NOT_FETCHED_YET'})
            continue
        source_status.append({'url':t['url'],'ids':t['ids'],'status':result['status'],
            'method':result['method'],'fetched_at':result['fetched_at'],'sha256':result['body_sha256']})
        if result['status']!='CONTENT_OK':
            left.append({'url':t['url'],'ids':t['ids'],'reason':result['status']});continue
        for key in t['ids']:
            row=id_lookup.get(key)
            if row is None:raise RuntimeError('Research source-join nicht eindeutig: '+key)
            for evidence in find_evidence(row['title'],result['body'],t['url']):
                candidates.append({'country_id':key,'source_checked_at':result['fetched_at'],
                    'source_body_sha256':result['body_sha256'],**evidence,
                    'decision':'CANDIDATE_ONLY_REQUIRES_OFFER_EDITION_FIELD_MATCH'})
    for url in reservations:
        if url not in paid_results:left.append({'url':url,'ids':byurl.get(url,{}).get('ids',[]),
                                                  'reason':'PAID_RESERVED_UNCERTAIN_NO_AUTO_RETRY'})
    summary={**core_report,'mode':MODUS,'free_results':len(free_results),
        'paid_reservations':len(reservations),'paid_results':len(paid_results),
        'credit_balance_before':credits_at_start,'credit_balance_after':credits_at_end,
        'exact_field_evidence':'NO_AUTOMATIC_FIELD_PROMOTION',
        'source_snippet_candidates':len(candidates),'unresolved_urls':len(left),
        'completed_at':datetime.now(timezone.utc).isoformat()}
    final=make_zip({'konsolidierter_3818_grunstock.jsonl':jsonl_bytes(all_3818),
        'quellenstatus.tsv':csv_bytes(source_status),
        'belegkandidaten_mit_originaltext.tsv':csv_bytes(candidates),
        'restfaelle_und_holds.tsv':csv_bytes(left),
        'historisch_gesperrt_145.jsonl':jsonl_bytes(bridge_locked),
        'stadt_nachweise_21.jsonl':jsonl_bytes(bridge_cities),
        'evt260_547_belegentscheide.csv':zip260['ALL_547_SUCCESSOR_SOURCE_DECISIONS.csv'],
        'kennzahlen.json':json.dumps(summary,ensure_ascii=False,indent=2)})
    output_name=f'ERGEBNIS_GROSSLAUF_{RUN_KEY}_{MODUS}_{sha(final)[:12]}.zip'
    write_immutable(output_name,final)
    print('\nFERTIG – Drive-Ordner:')
    print(f'https://drive.google.com/drive/folders/{folder_id}')
    print('Gesamtpaket:',output_name)
    print(json.dumps(summary,ensure_ascii=False,indent=2))
    display(pd.DataFrame([{'Datensätze':3818,'Quellenabrufe':len(free_results),
      'Firecrawl reserviert':len(reservations),'Originaltext-Kandidaten':len(candidates),
      'Noch offene Quellen':len(left)}]))


## Nach dem Knopfdruck

Das Notebook zeigt einen Google-Drive-Ordner mit **allen 3’818 zusammengeführten Country+ID-Zeilen**, Originalwerten, Preis-Sidecars, 547 bereits vorhandenen Evidenzentscheidungen, 145 gesperrten historischen Fällen, belegten 21 Städten, source-lined Kandidaten, Quellstatus und Restfällen.

**Den Drive-Ordner-Link hier einfügen.** Ich übernehme das Paket in der gleichen Konversation als Grundlage für feldgenaue Daten-Merges und Website-Gates, ohne die Primärquellen erneut abzurufen.

Keine Veröffentlichung durch dieses Notebook; die Website-Umstellung ist der nächste, separate kontrollierte Block. Nicht publizierte Anbieterinformationen bleiben als solche gekennzeichnet.
